# Airline delay：原始数据清单 / Raw data inventory

本 Notebook 展示原始下载清单与覆盖。完整重建、各阶段运行及已有档案的核对方法见项目根目录的 `docs/最终运行说明.md`；实际运行状态以 `experiments/index.csv` 和各 run 的 manifest 为准。

In [1]:
from pathlib import Path
cwd = Path.cwd().resolve()
ROOT = next((p for p in [cwd, *cwd.parents] if (p / 'scripts/build_features.py').is_file()), Path('D:\\airline-delay'))
RUN_DIR = ROOT / 'experiments/runs/R0031'
OUTPUT_RUN_DIR = ROOT / 'experiments/runs/R0031'
NOTEBOOK_DIR = ROOT / 'experiments/runs/R0031/notebooks'
RQ2_EDA_RUN_DIR = ROOT / 'experiments/runs/R0031'
MODEL_RUNS = []
RQ1_RUN_DIR = ROOT / 'experiments/runs/R0031'

In [2]:
from pathlib import Path
import json
import pandas as pd
ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
assert (ROOT / 'raw/flights').is_dir(), 'Open this notebook from the project root or notebooks folder.'

## 1. 已下载范围 / Downloaded scope

月份统计与天气覆盖来自下载完整性核对，不是清洗后样本量。

In [3]:
summary = json.loads((ROOT/'tables/download_summary.json').read_text(encoding='utf-8'))
print(json.dumps(summary, ensure_ascii=False, indent=2))
display(pd.read_csv(ROOT/'tables/flight_months.csv'))

{
  "retrieved_date_utc": "2026-10-23T17:59:36.607646+00:00",
  "flight_rows": 3057590,
  "airport_ids": 355,
  "flight_months": [
    "2026-04",
    "2026-05",
    "2026-06",
    "2026-07",
    "2026-08"
  ],
  "weather_start_utc": "2026-03-30T00:00:00Z",
  "weather_end_exclusive_utc": "2026-09-02T00:00:00Z",
  "weather": {
    "metar": {
      "raw_rows_with_buffer": 1635385,
      "airports_with_records": 355,
      "airports_all_five_months": 355
    },
    "taf": {
      "raw_rows_with_buffer": 1368924,
      "airports_with_records": 343,
      "airports_all_five_months": 343
    }
  },
  "weather_files": 36,
  "training_executed": false,
  "full_flight_cleaning_executed": false,
  "noaa": {
    "files": 25,
    "original_bytes": 654058715,
    "compressed_bytes": 24708216,
    "stations_full_153_dates": 19,
    "stations_with_missing_dates": [
      "PQI",
      "OTH",
      "GST",
      "GUC",
      "WYS",
      "STX"
    ]
  },
  "taf_text": {
    "unique_csv_product_ids": 3178

,month,flights,airlines,airports,unique_nonempty_tails,first_date,last_date,named_raw_columns,zip_MB
0,2026-04,597919,13,342,5823,2026-04-01,2026-04-30,109,30.504623
1,2026-05,611735,13,347,5721,2026-05-01,2026-05-31,109,31.716693
2,2026-06,607577,12,351,5654,2026-06-01,2026-06-30,109,31.606062
3,2026-07,631970,12,352,5692,2026-07-01,2026-07-31,109,33.075361
4,2026-08,608389,12,352,5709,2026-08-01,2026-08-31,109,32.031366


## 2. 逐机场逐月天气覆盖 / Weather coverage

`rows` 是源档案记录或预报分段数；`days` 是有记录的不同 UTC 日期数。TAF 的一天会发布多个版本，每个版本又含多个预报分段。

In [4]:
coverage = pd.read_csv(ROOT/'tables/weather_coverage.csv')
display(coverage.groupby(['kind','month']).agg(airports_with_records=('rows', lambda x: (x > 0).sum()), records=('rows','sum')).reset_index())
display(pd.read_csv(ROOT/'tables/weather_unavailable.csv'))

,kind,month,airports_with_records,records
0,metar,2026-04,355,320962
1,metar,2026-05,355,325365
2,metar,2026-06,355,315261
3,metar,2026-07,355,319179
4,metar,2026-08,355,323927
5,taf,2026-04,343,280088
6,taf,2026-05,343,276341
7,taf,2026-06,343,266259
8,taf,2026-07,343,255040
9,taf,2026-08,343,265099


,airport_id,airport_code,kind,rows,status
0,10165,ADK,taf,0,No records returned by IEM for 2026-03-30 to 2...
1,10676,BLV,taf,0,No records returned by IEM for 2026-03-30 to 2...
2,11587,ESC,taf,0,No records returned by IEM for 2026-03-30 to 2...
3,11982,GRK,taf,0,No records returned by IEM for 2026-03-30 to 2...
4,12119,HGR,taf,0,No records returned by IEM for 2026-03-30 to 2...
5,12124,HHH,taf,0,No records returned by IEM for 2026-03-30 to 2...
6,12335,IMT,taf,0,No records returned by IEM for 2026-03-30 to 2...
7,12544,USA,taf,0,No records returned by IEM for 2026-03-30 to 2...
8,13541,MVY,taf,0,No records returned by IEM for 2026-03-30 to 2...
9,14802,SHD,taf,0,No records returned by IEM for 2026-03-30 to 2...


## 3. 文件迁移核对 / Offline integrity check

在终端运行 `python scripts/verify_downloads.py --raw-only`。检查哈希与压缩文件完整性，不联网、不改写原始数据。

航班清洗与衔接证据见 `01_航班清洗与衔接.ipynb`；开发比较及最终评估见 06、07 Notebook。迁移时同时同步并校验原始输入、本地派生数据、模型和预测，不能把 Git 克隆单独视为完整研究备份。